# Telecom Egypt Intelligent Assistant (Self-Contained Groq Version)

This notebook contains the complete source code for the Telecom Egypt RAG pipeline, making it fully self-contained. You can run this directly on **Google Colab** or **Kaggle** without needing to upload the rest of the python scripts from the repository.

It uses the **Groq API** for fast, high-quality generation.

## 1. Install Dependencies

In [1]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers transformers edge-tts pypdf python-docx docx2txt pillow beautifulsoup4 requests langchain-groq groq

In [2]:
!pip install docx2txt

## 2. Setup API Key and Logging
Get your API key from the [Groq Console](https://console.groq.com/).

In [3]:
import os
import requests
from bs4 import BeautifulSoup
from typing import List
import logging
from PIL import Image
from getpass import getpass

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Please enter your Groq API Key:")
os.environ["GROQ_API_KEY"] = getpass()

Please enter your Groq API Key:
··········


## 3. Define Prompts

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

RAG_SYSTEM_PROMPT = """You are a helpful, direct, and intelligent assistant for Telecom Egypt (WE).
Your primary task is to answer the user's question based on the provided Knowledge Base Context, Uploaded Documents, and conversation history.

Follow these STRICT rules:
1. Answer the question using ONLY the provided Knowledge Base Context, Uploaded Documents, and conversation history. If the answer cannot be found in these sources, say strictly: "I do not have enough information to answer that based on the provided context." Do not use external knowledge.
2. DIRECT ANSWERS ONLY: DO NOT explain your reasoning, DO NOT mention what language the user is speaking, and DO NOT output meta-commentary like "Since the question is in English..." or "The uploaded document asks...". Give ONLY the final answer directly to the user.
3. CRITICAL LANGUAGE RULE: Detect the language of the user's input (or uploaded document) and reply in the exact same language. If Arabic, reply in Arabic. If English, reply in English. Do NOT translate the user's text; just respond in that language.
4. REASONING: All step-by-step thinking MUST be enclosed strictly inside <think>...</think> tags at the very beginning of your response. Never leak your thought process or language selection logic outside of these tags!
5. DOCUMENT PRIORITY: If an Uploaded Document Content is provided below, read it carefully. If it contains a question, answer that question using the Knowledge Base Context!

--- Knowledge Base Context ---
{context}

--- Uploaded Document Content ---
{uploaded_context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

## 4. Document Scrapers and Loaders

In [15]:
import base64
import os
from groq import Groq
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def process_image_ocr(file_path: str) -> list[Document]:
    try:
        with open(file_path, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode('utf-8')

        client = Groq()
        logger.info(f"Running high-quality OCR via Groq Vision for {file_path}...")
        completion = client.chat.completions.create(
            model="llama-3.2-90b-vision-instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Extract all text from this image exactly as written. Output ONLY the extracted text, with absolutely no commentary."},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                    ]
                }
            ],
            temperature=0,
            max_completion_tokens=2048,
        )
        text = completion.choices[0].message.content
        return [Document(page_content=text, metadata={"source": file_path, "type": "image_ocr"})]
    except Exception as e:
        logger.error(f"Error processing image {file_path}: {e}")
        return []

def load_document(file_path: str) -> list[Document]:
    if not os.path.exists(file_path):
        logger.error(f"File not found: {file_path}")
        return []

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == '.pdf':
            try:
                return PyPDFLoader(file_path).load()
            except Exception as pdf_err:
                logger.warning(f"PyPDFLoader failed ({pdf_err}), using direct pypdf fallback...")
                import pypdf
                reader = pypdf.PdfReader(file_path)
                text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
                return [Document(page_content=text, metadata={"source": file_path, "type": "pdf"})]
        elif ext == '.docx': return Docx2txtLoader(file_path).load()
        elif ext == '.txt': return TextLoader(file_path, encoding='utf-8').load()
        elif ext in ['.png', '.jpg', '.jpeg']: return process_image_ocr(file_path)
        else:
            logger.warning(f"Unsupported file extension: {ext}")
            return []
    except Exception as e:
        logger.error(f"Error loading {file_path}: {e}")
        return []

def scrape_te_page(url: str) -> list[Document]:
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'

        soup = BeautifulSoup(response.text, 'html.parser')
        for script in soup(["script", "style", "header", "footer", "nav"]):
            script.decompose()

        text = soup.get_text(separator=' ', strip=True)
        return [Document(page_content=text, metadata={"source": url, "type": "web_page"})]
    except Exception as e:
        logger.error(f"Error scraping {url}: {e}")
        return []

## 5. Vector Store Configuration

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def get_embeddings_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

def setup_vector_store(documents: List[Document], persist_directory: str = "./data/chroma_db") -> Chroma:
    if not documents:
        return Chroma(collection_name="te_knowledge_base", embedding_function=get_embeddings_model(), persist_directory=persist_directory)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
    chunks = text_splitter.split_documents(documents)
    logger.info(f"Split documents into {len(chunks)} chunks.")

    os.makedirs(persist_directory, exist_ok=True)
    return Chroma.from_documents(
        documents=chunks,
        embedding=get_embeddings_model(),
        persist_directory=persist_directory,
        collection_name="te_knowledge_base"
    )

## 6. Run Ingestion (Build Knowledge Base)

In [7]:
# List of high-value TE knowledge targets
faq_urls = [
    "https://www.te.eg/about-te/faq",                  # Main Mobile & USSD FAQs
    "https://te.eg/en/about-te/faq/fixed-broadband",   # Home Internet (WE Space, Routers, Quotas)
    "https://te.eg/en/about-te/faq/fixed-voice"        # Landline (Billing, Installments, Tariffs)
]

all_docs = []
for url in faq_urls:
    logger.info(f"Processing knowledge target: {url}")
    # Using our updated Jina AI scraper from Section 4
    docs = scrape_te_page(url)
    all_docs.extend(docs)

# Set up the Chroma DB and generate embeddings from all scraped FAQ pages
vector_db = setup_vector_store(all_docs, persist_directory="./data/chroma_db")
logger.info(f"Knowledge base successfully populated with {len(all_docs)} source pages!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 7. RAG Pipeline Implementation (Groq)

In [8]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma

def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source', 'Unknown')}]\nContent: {d.page_content}" for d in docs)

class GroqRAGPipeline:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile"):
        self.vector_store = Chroma(
            collection_name="te_knowledge_base",
            embedding_function=get_embeddings_model(),
            persist_directory="./data/chroma_db"
        )
        self.retriever = self.vector_store.as_retriever(search_kwargs={"k": 3})
        self.llm = ChatGroq(model_name=model_name, temperature=0.1)

        # FIXED: Feed both the user input AND the uploaded document text into the vector retriever!
        self.rag_chain = (
            {
                "context": lambda x: format_docs(self.retriever.invoke(f"{x['input']} {x.get('uploaded_context', '')}".strip()[:1000])),
                "uploaded_context": lambda x: x.get("uploaded_context", "No document currently uploaded."),
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"]
            }
            | qa_prompt
            | self.llm
            | StrOutputParser()
        )

    def query(self, user_input: str, chat_history: list = None, uploaded_context: str = "") -> dict:
        search_query = f"{user_input} {uploaded_context}".strip()[:1000]
        docs = self.retriever.invoke(search_query)
        answer = self.rag_chain.invoke({
            "input": user_input,
            "chat_history": chat_history or [],
            "uploaded_context": uploaded_context or "No document currently uploaded."
        })
        return {"answer": answer, "context": docs}

    def stream_query(self, user_input: str, chat_history: list = None, uploaded_context: str = ""):
        search_query = f"{user_input} {uploaded_context}".strip()[:1000]
        docs = self.retriever.invoke(search_query)
        stream = self.rag_chain.stream({
            "input": user_input,
            "chat_history": chat_history or [],
            "uploaded_context": uploaded_context or "No document currently uploaded."
        })
        for chunk in stream:
            yield chunk, docs

logger.info("Initializing Groq RAG Pipeline...")
pipeline = GroqRAGPipeline(model_name="llama-3.3-70b-versatile")
logger.info("Pipeline Ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 8. Query the Pipeline

In [9]:
query = "What services does Telecom Egypt offer for personal use?"
result = pipeline.query(query)

print("\nTE Assistant:")
print("-" * 60)
print(result["answer"])
print("-" * 60)

if result["context"]:
    print("\nSources:")
    sources = set(doc.metadata.get('source', 'Unknown') for doc in result["context"])
    for source in sources:
        print(f"  • {source}")


TE Assistant:
------------------------------------------------------------
<think>...</think>
Telecom Egypt offers services including mobile, fixed voice, and internet (both mobile and fixed broadband) for personal use.
------------------------------------------------------------

Sources:
  • https://te.eg/en/about-te/faq/fixed-broadband
  • https://www.te.eg/about-te/faq


## 9. Speech Integration (ASR & TTS)

In [16]:
import subprocess
import logging
from groq import Groq

logger = logging.getLogger(__name__)

class EgyptianASR:
    def __init__(self):
        self.client = Groq()

    def transcribe(self, audio_path: str) -> str:
        logger.info(f"Transcribing {audio_path} with Groq Whisper (Egyptian Arabic optimized)...")
        try:
            with open(audio_path, "rb") as file:
                transcription = self.client.audio.transcriptions.create(
                    file=(audio_path, file.read()),
                    # 1. Upgrade from turbo to large-v3 for superior Arabic dialect accuracy
                    model="whisper-large-v3",
                    # 3. Domain-specific Egyptian telecom prompt to guide the tokenizer
                    prompt="Telecom Egypt, WE, customer service, router, internet, المصرية للاتصالات، وي، فاتورة، باقة، رصيد، راوتر، إنترنت منزلي، سبيس.",                    # 4. Zero temperature eliminates random word guessing
                    temperature=0.0,
                    response_format="json",
                )
            return transcription.text
        except Exception as e:
            logger.error(f"ASR Transcription failed: {e}")
            return ""

class HighQualityTTS:
    def __init__(self, voice="ar-EG-SalmaNeural"):
        self.voice = voice

    def synthesize(self, text: str, output_path: str):
        logger.info(f"Synthesizing text using Edge TTS ({self.voice})")
        subprocess.run(
            ["edge-tts", "--voice", self.voice, "--text", text, "--write-media", output_path],
            check=True
        )

## 10. Test Audio Generation

In [11]:
from IPython.display import Audio

try:
    # Optional: Test ASR (Requires an uploaded audio file like "test_audio.wav")
    # asr = EgyptianASR()
    # transcript = asr.transcribe("test_audio.wav")
    # print(transcript)

    # Initialize TTS and synthesize the result
    tts = HighQualityTTS(voice="ar-EG-SalmaNeural")
    audio_file = "response_output.mp3"

    # Generate audio (using Colab's existing event loop)
    tts.synthesize(result["answer"], audio_file)

    logger.info("Generated audio successfully.")
except Exception as e:
    logger.error(f"Audio processing failed: {e}")

# Audio(audio_file) # Uncomment to play in notebook


## 11. Install Gradio

In [12]:
!pip install gradio


In [13]:
!pip install --upgrade gradio

## 12. Run the Interactive Frontend in the Notebook

In [ ]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage
import re
import uuid
import os
import base64

def wrap_rtl(text):
    if re.search("[؀-ۿ]", text):
        return f"<div dir='rtl' style='text-align: right;'>\n\n{text}\n\n</div>"
    return text

def get_clean_path(fp):
    """Safely extracts the file path from Gradio 4/5/6 file objects or dicts."""
    if isinstance(fp, dict):
        return fp.get("path") or fp.get("name") or str(fp)
    elif hasattr(fp, "path"):
        return getattr(fp, "path")
    elif hasattr(fp, "name"):
        return getattr(fp, "name")
    return str(fp)

def process_interaction(audio_filepath, file_paths, text_input, history, active_doc_text):
    if file_paths:
        active_doc_text = ""
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
        for fp in file_paths:
            actual_path = get_clean_path(fp)
            docs = load_document(actual_path)
            if docs:
                for d in docs:
                    text_content = d.page_content.strip() if d.page_content else ""
                    if text_content:
                        active_doc_text += f"\n[Uploaded Document: {os.path.basename(actual_path)}]\n{text_content}\n"
                chunks = text_splitter.split_documents(docs)
                if chunks: pipeline.vector_store.add_documents(chunks)

    if audio_filepath:
        asr = EgyptianASR()
        user_input = asr.transcribe(audio_filepath)
    else:
        user_input = text_input

    if not user_input and active_doc_text:
        user_input = "Analyze the uploaded document. If it asks a question about Telecom Egypt / WE, answer it directly using the Knowledge Base. If it is general text, provide a concise summary. Do not add introductory conversational filler."

    if not user_input:
        yield gr.update(value=""), gr.update(value=None), gr.update(), history, gr.update(), active_doc_text
        return

    display_user_input = user_input
    chat_history = []

    for msg in history:
        content = msg.get("content", "")
        if isinstance(content, (dict, list, tuple)): continue
        content_str = str(content)
        if content_str.startswith("{'path':") or "FileData" in content_str: continue

        if msg["role"] == "user":
            chat_history.append(HumanMessage(content=content_str))
        elif msg["role"] == "assistant":
            clean_content = content_str.replace("<div dir='rtl' style='text-align: right;'>\n\n", "").replace("\n\n</div>", "")
            chat_history.append(AIMessage(content=clean_content))

    prompt_doc_context = active_doc_text
    if len(prompt_doc_context) > 15000:
        prompt_doc_context = prompt_doc_context[:15000] + "\n... (truncated due to length)"

    if file_paths:
        for fp in file_paths:
            actual_path = get_clean_path(fp)
            history.append({"role": "user", "content": gr.FileData(path=actual_path)})

    history.append({"role": "user", "content": display_user_input})
    history.append({"role": "assistant", "content": ""})

    full_answer, docs = "", []
    for chunk, retrieved_docs in pipeline.stream_query(user_input, chat_history=chat_history, uploaded_context=prompt_doc_context):
        full_answer += chunk
        docs = retrieved_docs

        display_text = re.sub(r"&lt;think&gt;.*?(&lt;/think&gt;|$)", "", full_answer, flags=re.DOTALL | re.IGNORECASE)
        display_text = re.sub(r"<think[^>]*>.*?(</think[^>]*>|$)", "", display_text, flags=re.DOTALL | re.IGNORECASE).strip()
        history[-1]["content"] = display_text
        yield gr.update(value=""), gr.update(value=None), gr.update(), history, gr.update(), active_doc_text

    is_arabic = bool(re.search("[؀-ۿ]", full_answer))
    voice = "ar-EG-SalmaNeural" if is_arabic else "en-US-AriaNeural"
    tts = HighQualityTTS(voice=voice)

    out_audio = f"response_{uuid.uuid4().hex}.mp3"
    spoken_answer = re.sub(r"&lt;think&gt;.*?&lt;/think&gt;", "", full_answer, flags=re.DOTALL | re.IGNORECASE)
    spoken_answer = re.sub(r"<think[^>]*>.*?</think[^>]*>", "", spoken_answer, flags=re.DOTALL | re.IGNORECASE).strip()

    try:
        if spoken_answer: tts.synthesize(spoken_answer, out_audio)
    except Exception:
        out_audio = None

    answer_text = re.sub(r"&lt;think&gt;.*?(&lt;/think&gt;|$)", "", full_answer, flags=re.DOTALL | re.IGNORECASE)
    answer_text = re.sub(r"<think[^>]*>.*?(</think[^>]*>|$)", "", answer_text, flags=re.DOTALL | re.IGNORECASE).strip()
    history[-1]["content"] = wrap_rtl(answer_text)

    yield gr.update(value=""), gr.update(value=None), gr.update(value=None), history, out_audio, active_doc_text

we_theme = gr.themes.Soft(primary_hue="purple", secondary_hue="indigo").set(
    button_primary_background_fill="#5b2b82",
    button_primary_background_fill_hover="#4a226b",
    button_primary_text_color="white",
    block_title_text_color="#5b2b82"
)

try:
    with open("data/we_logo.png", "rb") as f:
        b64_logo = base64.b64encode(f.read()).decode("utf-8")
    logo_html = f"<div style='text-align: center;'><img src='data:image/png;base64,{b64_logo}' width='150' style='display: inline-block;'/></div>"
except Exception:
    logo_html = ""

with gr.Blocks(theme=we_theme, css=".gradio-container {max-width: 900px; margin: auto;} #sleek-audio { border-radius: 50px !important; box-shadow: 0px 4px 15px rgba(91, 43, 130, 0.2) !important; border: 2px solid #5b2b82 !important; overflow: hidden; margin-top: 10px; margin-bottom: 15px; } #sleek-audio .label-wrap { display: none !important; }") as demo:
    if logo_html:
        gr.HTML(logo_html)
    gr.Markdown("<h1 style='text-align: center; color: #5b2b82;'>Telecom Egypt Intelligent Assistant</h1>")

    doc_state = gr.State("")

    chatbot = gr.Chatbot(label="TE Assistant", height=500)
    with gr.Row():
        gr.HTML("<div style='flex-grow: 1;'></div>")
        with gr.Column(scale=2, min_width=250):
            audio_output = gr.Audio(autoplay=True, interactive=False, elem_id="sleek-audio")
        gr.HTML("<div style='flex-grow: 1;'></div>")

    with gr.Row():
        with gr.Column(scale=8):
            txt = gr.Textbox(show_label=False, placeholder="Type your message here...", container=False)
        with gr.Column(scale=1, min_width=80):
            submit_btn = gr.Button("Send", variant="primary")

    with gr.Row():
        audio_in = gr.Audio(sources=["microphone"], type="filepath", label="Record Voice (Optional)")
        file_in = gr.File(label="Attach Documents (Optional)", file_count="multiple", file_types=[".pdf", ".docx", ".txt", ".png", ".jpg", ".jpeg"], type="filepath")

    submit_btn.click(
        fn=lambda: gr.update(interactive=False), outputs=[submit_btn]
    ).then(
        fn=process_interaction,
        inputs=[audio_in, file_in, txt, chatbot, doc_state],
        outputs=[txt, audio_in, file_in, chatbot, audio_output, doc_state]
    ).then(
        fn=lambda: gr.update(interactive=True), outputs=[submit_btn]
    )

    txt.submit(
        fn=lambda: gr.update(interactive=False), outputs=[submit_btn]
    ).then(
        fn=process_interaction,
        inputs=[audio_in, file_in, txt, chatbot, doc_state],
        outputs=[txt, audio_in, file_in, chatbot, audio_output, doc_state]
    ).then(
        fn=lambda: gr.update(interactive=True), outputs=[submit_btn]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_7920/2118438435.py:122: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=we_theme, css=".gradio-container {max-width: 900px; margin: auto;} #sleek-audio { border-radius: 50px !important; box-shadow: 0px 4px 15px rgba(91, 43, 130, 0.2) !important; border: 2px solid #5b2b82 !important; overflow: hidden; margin-top: 10px; margin-bottom: 15px; } #sleek-audio .label-wrap { display: none !important; }") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a98dc6fa5de8ab9113.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
